# Session 3 — Data Quality Assessment · SOLUTIONS
**Pandas & Data Manipulation** · Albert School × Politecnico di Milano

| Part | What you do | Time |
|---|---|---|
| **1. Demo** | Run each cell: the examples of the slides, on the tiny table | 10 min |
| **2. Practice** | 10 questions on real taxi data, from ★ to ★★★ | 60 min |

**Today we assess, we do not clean.** Every check returns a True/False **mask**:
`mask.sum()` → how many rows · `mask.mean()` → which share.

| Dimension | Question | Main tools |
|---|---|---|
| 🟦 Missing values | Is the value there? | `isna()`, `value_counts()` |
| 🟧 Inconsistencies | Is it written one way? | `unique()`, `nunique()`, `.str`, `pd.to_numeric()` |
| 🟩 Duplication | Is each fact recorded once? | `duplicated()`, `subset=`, `keep=` |
| 🟨 Accuracy | Is the value plausible? | `describe()`, `between()`, `isin()`, `quantile()` |

## Part 1 — Demo
### 0 · The tiny table from the slides
It hides all four problems: missing cells, a placeholder (`"unknown"`), two spellings of a country,
a duplicated order, an impossible age and a negative total.

In [1]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "order_id": [101, 102, 103, 104, 101, 105],
    "customer": ["Ana", "Ben", None, "Dan", "Ana", "Eva"],
    "country":  ["FR", "it", "IT", "FR", "FR", " ES"],
    "age":      [34, None, 29, 230, 34, 41],
    "total":    [120, 80, -50, 95, 120, None],
    "status":   ["shipped", "delivered", "shipped", "unknown", "shipped", "returned"],
})
df

,order_id,customer,country,age,total,status
0,101,Ana,FR,34.0,120.0,shipped
1,102,Ben,it,NaN,80.0,delivered
2,103,NaN,IT,29.0,-50.0,shipped
3,104,Dan,FR,230.0,95.0,unknown
4,101,Ana,FR,34.0,120.0,shipped
5,105,Eva,ES,41.0,NaN,returned


### 1 · Missing values
**NaN** (*Not a Number*) marks an empty cell. Statistics skip it silently, and it is never equal to anything.

In [2]:
s = pd.Series([10, None, 30])
print(s.tolist(), s.dtype)          # None became NaN, the integers became floats
print(len(s), s.count())            # 3 values, 2 not missing
print(s.mean())                     # 20.0 = (10 + 30) / 2: NaN is skipped
print(np.nan == np.nan)             # False -> never test "== np.nan", use isna()

[10.0, nan, 30.0] float64
3 2
20.0
False


`df.isna()` → a True/False map of the whole table (True = missing). `df.notna()` is the opposite.

In [3]:
df.isna()

,order_id,customer,country,age,total,status
0,False,False,False,False,False,False
1,False,False,False,True,False,False
2,False,True,False,False,False,False
3,False,False,False,False,False,False
4,False,False,False,False,False,False
5,False,False,False,False,True,False


True counts as 1 and False as 0: `sum()` counts the missing cells, `mean()` gives the share.

In [4]:
pd.DataFrame({
    "n_missing": df.isna().sum(),          # how many per column
    "pct": df.isna().mean() * 100,         # which share per column (%)
}).round(1)

,n_missing,pct
order_id,0,0.0
customer,1,16.7
country,0,0.0
age,1,16.7
total,1,16.7
status,0,0.0


`axis=1` works across each row: `any(axis=1)` is True when a row has at least one NaN.

In [5]:
incomplete = df.isna().any(axis=1)
print(incomplete.sum(), "incomplete rows")
df[incomplete]

3 incomplete rows


,order_id,customer,country,age,total,status
1,102,Ben,it,NaN,80.0,delivered
2,103,NaN,IT,29.0,-50.0,shipped
5,105,Eva,ES,41.0,NaN,returned


In [6]:
df.info()      # the Non-Null Count column gives the same information

<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   order_id  6 non-null      int64  
 1   customer  5 non-null      str    
 2   country   6 non-null      str    
 3   age       5 non-null      float64
 4   total     5 non-null      float64
 5   status    6 non-null      str    
dtypes: float64(2), int64(1), str(3)
memory usage: 420.0 bytes


**Placeholders** mean "we do not know" but are stored as text or numbers: `isna()` does not see them.

In [7]:
print(df["status"].isna().sum())     # 0 ...
df["status"].value_counts()          # ... but "unknown" is a missing value in disguise

0


status
shipped      3
delivered    1
unknown      1
returned     1
Name: count, dtype: int64

`read_csv` turns empty cells, `N/A`, `NA`, `null` into NaN by default. Declare other placeholders with `na_values=`.

In [8]:
from io import StringIO

csv = "city,temp\nParis,12\nRome,N/A\nMilan,unknown\nLyon,\n"
d1 = pd.read_csv(StringIO(csv))
d2 = pd.read_csv(StringIO(csv), na_values=["unknown"])
print("default   :", d1["temp"].isna().sum(), "missing, dtype", d1["temp"].dtype)
print("na_values :", d2["temp"].isna().sum(), "missing, dtype", d2["temp"].dtype)

default   : 2 missing, dtype str
na_values : 3 missing, dtype float64


### 2 · Inconsistencies
`unique()` lists the distinct values, `nunique()` counts them, `value_counts()` gives each value with its frequency.

In [9]:
print(df["country"].unique())
print(df["country"].nunique(), "different strings")
df["country"].value_counts()

<StringArray>
['FR', 'it', 'IT', ' ES']
Length: 4, dtype: str
4 different strings


country
FR     3
it     1
IT     1
 ES    1
Name: count, dtype: int64

Normalise a **copy** (`.str.strip()` removes spaces, `.str.upper()` sets one case) to measure the variants. We do **not** write it back into `df`.

In [10]:
norm = df["country"].str.strip().str.upper()
print(norm.nunique(), "countries after normalising")
print((df["country"] != norm).sum(), "rows written differently")
pd.DataFrame({"country": df["country"], "normalised": norm, "differs": df["country"] != norm})

3 countries after normalising
2 rows written differently


,country,normalised,differs
0,FR,FR,False
1,it,IT,True
2,IT,IT,False
3,FR,FR,False
4,FR,FR,False
5,ES,ES,True


A numeric column stored as text hides at least one non-number. `pd.to_numeric(errors="coerce")` turns non-numbers into NaN.

In [11]:
p = pd.Series(["10", "12.5", "n.d.", "8"])
num = pd.to_numeric(p, errors="coerce")
print(p.dtype, "->", num.dtype)
print(num.isna().sum(), "value is not a number:", p[num.isna()].tolist())

str -> float64
1 value is not a number: ['n.d.']


Related columns must agree: here `total` should be `qty * price`.

In [12]:
o = pd.DataFrame({"qty": [2, 1, 3, 2],
                  "price": [5.0, 20.0, 4.0, 10.0],
                  "total": [10.0, 20.0, 15.0, 20.0]})
bad = o["total"] != o["qty"] * o["price"]
print(bad.sum(), "row breaks the rule")
o[bad]

1 row breaks the rule


,qty,price,total
2,3,4.0,15.0


### 3 · Duplication
`duplicated()` is True for every row identical to an **earlier** row.

In [13]:
print(df.duplicated().sum(), "duplicate row")
df[df.duplicated(keep=False)]       # keep=False -> show every copy

1 duplicate row


,order_id,customer,country,age,total,status
0,101,Ana,FR,34.0,120.0,shipped
4,101,Ana,FR,34.0,120.0,shipped


`subset=` checks only a key. Here two rows share `id = 2` but have different prices.

In [14]:
k = pd.DataFrame({"id": [1, 2, 2, 3],
                  "item": ["pen", "cup", "cup", "mug"],
                  "price": [2.0, 5.0, 5.5, 8.0]})
print(k.duplicated().sum(), "exact duplicates")
print(k.duplicated(subset="id").sum(), "repeated id")
print(k["id"].is_unique)

0 exact duplicates
1 repeated id
False


### 4 · Accuracy
`describe()` shows min and max: impossible extremes jump out.

In [15]:
df[["age", "total"]].describe()

,age,total
count,5.000000,5.000000
mean,73.600000,73.000000
std,87.534565,70.851958
min,29.000000,-50.000000
25%,34.000000,80.000000
50%,34.000000,95.000000
75%,41.000000,120.000000
max,230.000000,120.000000


`between(a, b)` tests a range (both ends included). NaN is never in the range, so keep only existing values with `notna()`.

In [16]:
ok = df["age"].between(0, 110)
print((~ok).sum(), "<- wrong: also counts the NaN")
bad_age = df["age"].notna() & ~ok
print(bad_age.sum(), "<- right: one impossible age")
df[bad_age]

2 <- wrong: also counts the NaN
1 <- right: one impossible age


,order_id,customer,country,age,total,status
3,104,Dan,FR,230.0,95.0,unknown


`isin(list)` tests a set of allowed values; `~` reverses the mask.

In [17]:
allowed = ["shipped", "delivered", "returned", "cancelled"]
not_allowed = ~df["status"].isin(allowed)
df.loc[not_allowed, "status"]

3    unknown
Name: status, dtype: str

IQR rule: values far outside the middle 50% are **suspicious**, not automatically wrong.

In [18]:
days = pd.Series([2, 3, 3, 4, 5, 4, 30])
q1, q3 = days.quantile(0.25), days.quantile(0.75)
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(q1, q3, iqr, low, high)
days[(days < low) | (days > high)]

3.0 4.5 1.5 0.75 6.75


6    30
dtype: int64

### 5 · The quality report
One named mask per check, then one row per check.

In [19]:
miss  = df.isna().any(axis=1)
unk   = df["status"] == "unknown"
spell = df["country"] != norm
dup   = df.duplicated()
neg   = df["total"] < 0

report = pd.DataFrame({
    "check": ["missing value", "status unknown", "country not standard",
              "duplicate row", "age outside 0-110", "total below 0"],
    "rows": [miss.sum(), unk.sum(), spell.sum(), dup.sum(), bad_age.sum(), neg.sum()],
})
report["pct"] = (report["rows"] / len(df) * 100).round(1)
report

,check,rows,pct
0,missing value,3,50.0
1,status unknown,1,16.7
2,country not standard,2,33.3
3,duplicate row,1,16.7
4,age outside 0-110,1,16.7
5,total below 0,1,16.7


## Part 2 — Practice (60 minutes)

**The scenario.** The city mobility office from Session 2 received a new export of the March 2019 taxi rides.
Before anyone computes statistics, your manager asks you for a **data quality assessment**:
*what is wrong with this file, where, and how much?*

**Rules**
- **Do not modify `df`.** You only look and count; cleaning comes in the next sessions.
- One empty cell per question. Add cells freely.
- End each answer with a `#` comment that states the result **in words**, e.g. `# 44 rides have no payment`.
- Keep your masks in variables with clear names: you will reuse them in Q10.

**Dataset:** `data/taxis_dirty.csv` — 15 columns: `ride_id` (one id per ride), `pickup`, `dropoff`, `passengers`, `distance`,
`fare`, `tip`, `tolls`, `total`, `color`, `payment`, `pickup_zone`, `dropoff_zone`, `pickup_borough`, `dropoff_borough`.
Real problems plus a few added on purpose: you do not know how many.

In [20]:
import pandas as pd

# The data folder must sit next to this notebook
df = pd.read_csv("data/taxis_dirty.csv", parse_dates=["pickup", "dropoff"])
df.head()

,ride_id,pickup,dropoff,passengers,distance,fare,tip,tolls,total,color,payment,pickup_zone,dropoff_zone,pickup_borough,dropoff_borough
0,18,2019-03-23 20:50:49,2019-03-23 21:02:07,1,2.60,10.5,2.00,0.00,16.30,yellow,credit card,Midtown Center,East Harlem South,Manhattan,Manhattan
1,3078,2019-03-14 02:45:24,2019-03-14 03:13:13,1,17.95,52.0,0.00,0.00,55.30,yellow,cash,JFK Airport,Midtown North,Queens,Manhattan
2,1586,2019-03-17 11:21:51,2019-03-17 11:29:43,2,1.72,8.0,2.82,0.00,14.12,yellow,credit card,Yorkville West,Midtown East,Manhattan,Manhattan
3,4937,2019-03-06 17:15:14,2019-03-06 17:19:16,3,0.70,5.0,0.00,0.00,9.30,yellow,cash,Lincoln Square West,Lincoln Square East,Manhattan,Manhattan
4,1785,2019-03-13 16:58:43,2019-03-13 18:04:48,5,19.25,52.0,13.11,5.76,78.67,yellow,credit card,Lenox Hill East,JFK Airport,Manhattan,Queens


### ★ Warm-up

**★ · Q1 — First look.**
1. How many rows and columns does the table have?
2. Print the dtype of every column. Which columns hold numbers, which hold text, which hold dates?

In [21]:
print(df.shape)
df.dtypes
# 6,483 rows and 15 columns.
# Numbers: ride_id, passengers, distance, fare, tip, tolls, total.
# Dates: pickup, dropoff. Text: color, payment, the zones and the boroughs.

(6483, 15)


ride_id                     int64
pickup             datetime64[us]
dropoff            datetime64[us]
passengers                  int64
distance                  float64
fare                      float64
tip                       float64
tolls                     float64
total                     float64
color                         str
payment                       str
pickup_zone                   str
dropoff_zone                  str
pickup_borough                str
dropoff_borough               str
dtype: object

**★ · Q2 — Missing cells per column.**
1. How many missing values does each column have?
2. Which column has the most? How many missing cells are there in the whole table?

In [22]:
print(df.isna().sum())
print(df.isna().sum().sum())
# dropoff_zone and dropoff_borough have the most (46 each); payment has 44.
# 190 missing cells in total, all in text columns.

ride_id             0
pickup              0
dropoff             0
passengers          0
distance            0
fare                0
tip                 0
tolls               0
total               0
color               0
payment            44
pickup_zone        27
dropoff_zone       46
pickup_borough     27
dropoff_borough    46
dtype: int64
190


**★ · Q3 — Share of missing values.**
1. Compute the **percentage** of missing values per column, rounded to 2 decimals.
2. How many rows have **at least one** missing value? *(Hint: `any(axis=1)`)*

In [23]:
print((df.isna().mean() * 100).round(2))
miss = df.isna().any(axis=1)
print(miss.sum())
# No column misses more than 0.71% of its values. 93 rows (1.4%) have at least one missing cell.

ride_id            0.00
pickup             0.00
dropoff            0.00
passengers         0.00
distance           0.00
fare               0.00
tip                0.00
tolls              0.00
total              0.00
color              0.00
payment            0.68
pickup_zone        0.42
dropoff_zone       0.71
pickup_borough     0.42
dropoff_borough    0.71
dtype: float64
93


### ★★ Look closer

**★★ · Q4 — Missing values in disguise.**
1. Show the `value_counts()` of `payment`, **including** NaN (`dropna=False`).
2. Which value is a placeholder for a missing payment? How many rides have it?
3. How many rides have **no usable payment** in total (NaN + placeholder)?

In [24]:
print(df["payment"].value_counts(dropna=False))
unknown = df["payment"] == "unknown"
print(unknown.sum())
print(df["payment"].isna().sum() + unknown.sum())
# "unknown" is a placeholder: 30 rides. With the 44 NaN, 74 rides have no usable payment.

payment
credit card     4497
cash            1791
NaN               44
Credit card       34
CREDIT CARD       31
unknown           30
credit card       27
CASH              11
 cash              9
Cash               9
Name: count, dtype: int64
30
74


**★★ · Q5 — One payment, many spellings.**
1. How many distinct values does `payment` have (`nunique()`)? How many after `.str.strip().str.lower()`?
2. How many rides have a payment written in a non-standard way?
   *Careful: NaN is never equal to anything, so keep only the payments that exist (`notna()`).*
3. Look at the `value_counts()` of `pickup_borough`: do you see a spelling problem there too?

In [25]:
norm = df["payment"].str.strip().str.lower()
print(df["payment"].nunique(), norm.nunique())
spelling = df["payment"].notna() & (df["payment"] != norm)
print(spelling.sum())
print(df["pickup_borough"].value_counts(dropna=False))
# 9 spellings for only 3 real values (cash, credit card, unknown).
# 121 rides have a non-standard spelling (e.g. "CASH", " cash", "credit card " with a trailing space).
# pickup_borough: "manhattan" (25 rides) is written in lower case next to "Manhattan".

9 3
121
pickup_borough
Manhattan    5286
Queens        660
Brooklyn      386
Bronx          99
NaN            27
manhattan      25
Name: count, dtype: int64


**★★ · Q6 — Exact duplicates.**
1. How many rows are exact copies of an earlier row?
2. Show **all** copies of the duplicated rows, sorted by `ride_id` (the first 6 rows are enough).

In [26]:
dup = df.duplicated()
print(dup.sum())
df[df.duplicated(keep=False)].sort_values("ride_id").head(6)
# 40 rows are exact copies of another row: every value is identical.

40


,ride_id,pickup,dropoff,passengers,distance,fare,tip,tolls,total,color,payment,pickup_zone,dropoff_zone,pickup_borough,dropoff_borough
73,42,2019-03-24 18:42:56,2019-03-24 19:09:00,0,4.5,20.0,0.00,0.0,23.30,yellow,cash,TriBeCa/Civic Center,Kips Bay,Manhattan,Manhattan
3047,42,2019-03-24 18:42:56,2019-03-24 19:09:00,0,4.5,20.0,0.00,0.0,23.30,yellow,cash,TriBeCa/Civic Center,Kips Bay,Manhattan,Manhattan
1674,78,2019-03-05 15:41:23,2019-03-05 15:46:30,1,0.6,5.0,1.65,0.0,9.95,yellow,credit card,Midtown Center,Midtown Center,Manhattan,Manhattan
543,78,2019-03-05 15:41:23,2019-03-05 15:46:30,1,0.6,5.0,1.65,0.0,9.95,yellow,credit card,Midtown Center,Midtown Center,Manhattan,Manhattan
1097,951,2019-03-10 14:10:22,2019-03-10 14:19:28,1,0.6,7.0,1.00,0.0,11.30,yellow,credit card,Clinton East,Clinton West,Manhattan,Manhattan
5751,951,2019-03-10 14:10:22,2019-03-10 14:19:28,1,0.6,7.0,1.00,0.0,11.30,yellow,credit card,Clinton East,Clinton West,Manhattan,Manhattan


**★★ · Q7 — The same ride twice?**
`ride_id` should identify one ride.
1. Is `ride_id` unique?
2. How many rows repeat a `ride_id` that already appeared?
3. How many of them are **not** exact duplicates (same id, different values)?

In [27]:
print(df["ride_id"].is_unique)
same_id = df.duplicated(subset="ride_id")
print(same_id.sum())
print(same_id.sum() - dup.sum())
# ride_id is not unique: 50 rows repeat an id. 40 are exact copies (Q6),
# 10 have the same id but different values (another tip and total): which one is right?

False
50
10


### ★★★ Rules and report

**★★★ · Q8 — Impossible values.**
1. Run `describe()` on `passengers`, `distance`, `fare`, `tip`, `tolls`, `total`. Which min or max looks wrong?
2. Count the rides with a **negative fare**.
3. Count the rides with a number of passengers **outside 1–6**. *(Hint: `between()`)*
4. Count the rides with a **distance of 0**.

In [28]:
print(df[["passengers", "distance", "fare", "tip", "tolls", "total"]].describe().round(2))
neg_fare = df["fare"] < 0
bad_pass = ~df["passengers"].between(1, 6)
zero_dist = df["distance"] == 0
print(neg_fare.sum(), bad_pass.sum(), zero_dist.sum())
# min fare -21.0 and max passengers 9 are impossible; min passengers and min distance are 0.
# 8 negative fares, 102 rides with 0 or more than 6 passengers, 52 rides with distance 0.

       passengers  distance     fare      tip    tolls    total
count     6483.00   6483.00  6483.00  6483.00  6483.00  6483.00
mean         1.55      3.02    13.07     1.98     0.33    18.52
std          1.22      3.83    11.58     2.45     1.41    13.81
min          0.00      0.00   -21.00     0.00     0.00     1.30
25%          1.00      0.98     6.50     0.00     0.00    10.80
50%          1.00      1.63     9.50     1.70     0.00    14.16
75%          2.00      3.21    15.00     2.80     0.00    20.30
max          9.00     36.70   150.00    33.20    24.02   174.82
8 102 52


**★★★ · Q9 — Do the dates agree?**
1. How many rides **end before they start** (`dropoff` earlier than `pickup`)?
2. The file should contain only March 2019. How many rides were picked up in another month? *(Hint: `.dt.month`)*

In [29]:
before = df["dropoff"] < df["pickup"]
not_march = df["pickup"].dt.month != 3
print(before.sum(), not_march.sum())
df.loc[not_march, ["ride_id", "pickup", "dropoff"]]
# 6 rides end before they start; 1 ride was picked up on 28 February 2019, just before midnight.

6 1


,ride_id,pickup,dropoff
1517,6204,2019-02-28 23:29:03,2019-02-28 23:32:35


**★★★ · Q10 — The quality report.**
Build a DataFrame `report` with one row per check you made in Q3–Q9 (at least 8 checks) and two columns:
`rows` (how many rows fail the check) and `pct` (share of all rows, in %, 2 decimals).
Sort it from the most to the least frequent problem. Which problem is the most common?

In [30]:
report = pd.DataFrame({
    "check": ["missing value", "payment unknown", "payment spelling",
              "exact duplicate", "repeated ride_id", "negative fare",
              "passengers outside 1-6", "distance 0",
              "dropoff before pickup", "pickup not in March"],
    "rows": [miss.sum(), unknown.sum(), spelling.sum(),
             dup.sum(), same_id.sum(), neg_fare.sum(),
             bad_pass.sum(), zero_dist.sum(),
             before.sum(), not_march.sum()],
})
report["pct"] = (report["rows"] / len(df) * 100).round(2)
report.sort_values("rows", ascending=False)
# The most common problem is the spelling of payment (121 rows, 1.87%), then passengers outside 1-6 (102).
# Every problem touches less than 2% of the rows, but together they change counts, sums and averages.

,check,rows,pct
2,payment spelling,121,1.87
6,passengers outside 1-6,102,1.57
0,missing value,93,1.43
7,distance 0,52,0.80
4,repeated ride_id,50,0.77
3,exact duplicate,40,0.62
1,payment unknown,30,0.46
5,negative fare,8,0.12
8,dropoff before pickup,6,0.09
9,pickup not in March,1,0.02
